# Script for combining, cleaning, and inspecting Tweede Kamer data

In [16]:
# imports
import pandas as pd
import ast
from IPython.display import display, HTML
import pandas as pd
import re
from html import escape

In [17]:
def deduplicate_block(text):
    if not isinstance(text, str):
        return text
    sentences = text.split(". ")  # splits op zinnen
    unique_sentences = list(dict.fromkeys(sentences))  # behoud volgorde, verwijder dubbels
    return ". ".join(unique_sentences)



In [18]:
beleidsstukken = pd.read_csv("data/beleidsnotas_rijksoverheid_ai_related.csv", index_col=0)

# in column frontenddate, convert yyyy-..... to just year
beleidsstukken['frontenddate'] = beleidsstukken['frontenddate'].str.extract(r'(\d{4})')


beleidsstukken = beleidsstukken.rename(columns={"matched_keywords": "matched_keywords_all"})
beleidsstukken = beleidsstukken.rename(columns={"relevant_text": "body"})

# rename to year
beleidsstukken.rename(columns={'frontenddate': 'year'}, inplace=True)

#remove rows with year 2026
beleidsstukken = beleidsstukken[beleidsstukken['year'] != '2026']


beleidsstukken.drop(['introduction', 'canonical', 'dataurl', 'lastmodified', 'available', 'pdf_text'], axis=1, inplace=True)

#reset index
beleidsstukken = beleidsstukken.reset_index(drop=True)




# show updates
beleidsstukken.head()

,title,year,ai_related,company_hits,body,matched_keywords_all,type
0,Investeren in Perspectief (Beleidsnota 2018),2018,yes,"['adyen', 'google', 'x']",Ook biedt de agenda kansen aan het bedrijfslev...,"['adyen', 'google', 'x', 'kunstmatige intellig...",beleidsnota
1,Nota Defensie Industrie Strategie,2018,yes,[],Nederland wil zelf aan militaire kennisontwikk...,"['ai', 'artificiële intelligentie', 'drones', ...",beleidsnota
2,Nationale Strategie Digitaal Erfgoed 2021-2024,2021,yes,['google'],Zo wordt in de vernieuwde strategie nu ook de ...,"['google', 'ai', 'algoritmes', 'artificiële in...",beleidsnota
3,Beslisnota's bij de Kamerbrief over toekomstig...,2021,yes,['x'],25 januari\n37 9-2-2023 Nota-StasGB-Memo box 3...,"['x', 'ai']",beleidsnota
4,Beslisnota bij Kamerbrief over aanpak belastin...,2021,yes,['x'],Dit betreft de nota’s in de onderstaande tabel...,"['x', 'ai']",beleidsnota


In [19]:
beleidsstukken['year'].unique()

<StringArray>
['2018', '2021', '2022', '2023', '2024', '2025']
Length: 6, dtype: str

In [20]:
vergaderstukken = pd.read_csv("data/vergaderstukken_rijksoverheid_ai_related.csv", index_col=0)

# Reset index
vergaderstukken = vergaderstukken.reset_index(drop=True)

# Extract year from frontenddate
vergaderstukken['frontenddate'] = vergaderstukken['frontenddate'].str.extract(r'(\d{4})')

# Rename columns
vergaderstukken = vergaderstukken.rename(columns={
    "matched_keywords": "matched_keywords_all",
    "relevant_text": "body",
    "frontenddate": "year",   # ← gewoon meteen in één rename
})

# Drop columns
vergaderstukken = vergaderstukken.drop(
    ['introduction', 'canonical', 'dataurl', 'lastmodified', 'available', 'pdf_text'],
    axis=1
)

vergaderstukken.head()

,title,year,ai_related,company_hits,body,matched_keywords_all,type
0,Geannoteerde besluitenlijst ministerraad 28 me...,2021,yes,[],Conclusies van de coördinatiecommissie d.d. 25...,['kunstmatige intelligentie'],vergaderstuk
1,Agenda ministerraad 4 juni 2021,2021,yes,[],Programma Landelijke Vreemdelingen Voorziening...,"['algoritmen', 'artificiële intelligentie']",vergaderstuk
2,Geannoteerde besluitenlijst ministerraad 4 jun...,2021,yes,[],"1 juni 2021,\nnr.22 (Minister van BZ)\nDe conc...","['ai', 'algoritmen', 'artificiële intelligenti...",vergaderstuk
3,Geannoteerde besluitenlijst ministerraad 29 ok...,2021,yes,['x'],4. EU-implementatie\na. Wijziging van het Alge...,"['x', 'bard']",vergaderstuk
4,Geannoteerde besluitenlijst ministerraad 26 no...,2021,yes,[],Raad Buitenlandse Zaken (Handel) d.d. 29 novem...,"['ai', 'artificial intelligence']",vergaderstuk


In [21]:
vergaderstukken['year'].unique()

<StringArray>
['2021', '2022', '2023', '2024', '2025']
Length: 5, dtype: str

In [22]:
plenaire_verslagen = pd.read_csv(r"C:\Users\joly-\Github\HUMAN\tweede_kamer\scraping_plen_ver\plenaire_verslagen_classified.csv", index_col=0)
plenaire_verslagen.head()

,ai_related,matched_keywords_title,matched_keywords_body,matched_keywords_all,n_hits_title_total,n_hits_body_total,company_hits,relevant_text
filename,,,,,,,,
kamerstukken-plenaire_verslagen-detail-2014-2015-101_9ae18352.html,yes,[],['drones'],['drones'],0,1062,[],"Zij krijgt nr. 186 (32637). De Kamer,\ngehoord..."
kamerstukken-plenaire_verslagen-detail-2014-2015-103_09e01e2b.html,yes,[],['drones'],"['x', 'drones']",0,209,['x'],De voorzitter:Ik constateer dat de aanwezige l...
kamerstukken-plenaire_verslagen-detail-2014-2015-48_64f423b7.html,yes,[],['gemini'],"['x', 'gemini']",0,1060,['x'],"Daarover kunnen we van mening verschillen, maa..."
kamerstukken-plenaire_verslagen-detail-2014-2015-67_e1c6ae74.html,yes,[],['gemini'],"['asml', 'gemini']",0,185,['asml'],Ik heb daar met de collega-woordvoerder van me...
kamerstukken-plenaire_verslagen-detail-2015-2016-100_cce9e53b.html,yes,[],['algoritmen'],"['google', 'x', 'algoritmen']",0,274,"['google', 'x']",De voorzitter:Dan hebt u het woord. Mevrouw Yü...


In [23]:
plenaire_verslagen = pd.read_csv(r"C:\Users\joly-\Github\HUMAN\tweede_kamer\scraping_plen_ver\plenaire_verslagen_classified.csv", index_col=0)

# extract year from filename index before it gets dropped
# .values strips the mismatched RangeIndex so pandas doesn't align by label
plenaire_verslagen["year"] = plenaire_verslagen.index.str.extract(r'detail-(\d{4})-\d{4}')[0].values

plenaire_verslagen = plenaire_verslagen.reset_index(drop=True)

# drop columns
plenaire_verslagen.drop(['matched_keywords_title', 'matched_keywords_body', 'n_hits_title_total', 'n_hits_body_total'], axis=1, inplace=True)

plenaire_verslagen['type'] = 'plenair verslag'

# plenaire_verslagen = plenaire_verslagen.rename(columns={"filename": "title"} )
plenaire_verslagen = plenaire_verslagen.rename(columns={"relevant_text": "body"})

# plenaire_verslagen["year"] = plenaire_verslagen["title"].str.extract(r"(\d{4})")
plenaire_verslagen["body"] = plenaire_verslagen["body"].apply(deduplicate_block)

plenaire_verslagen.head()

,ai_related,matched_keywords_all,company_hits,body,year,type
0,yes,['drones'],[],"Zij krijgt nr. 186 (32637). De Kamer,\ngehoord...",2014,plenair verslag
1,yes,"['x', 'drones']",['x'],De voorzitter:Ik constateer dat de aanwezige l...,2014,plenair verslag
2,yes,"['x', 'gemini']",['x'],"Daarover kunnen we van mening verschillen, maa...",2014,plenair verslag
3,yes,"['asml', 'gemini']",['asml'],Ik heb daar met de collega-woordvoerder van me...,2014,plenair verslag
4,yes,"['google', 'x', 'algoritmen']","['google', 'x']",De voorzitter:Dan hebt u het woord. Mevrouw Yü...,2015,plenair verslag


In [24]:
# combine dataframes
tweede_kamer_data = pd.concat([beleidsstukken, vergaderstukken, plenaire_verslagen], ignore_index=True)
# tweede_kamer_data = tweede_kamer_data.drop(columns=['ai_related'])
#rename  column
# tweede_kamer_data = tweede_kamer_data.rename(columns={"word_count": "original_word_count"})
tweede_kamer_data.head()

,title,year,ai_related,company_hits,body,matched_keywords_all,type
0,Investeren in Perspectief (Beleidsnota 2018),2018,yes,"['adyen', 'google', 'x']",Ook biedt de agenda kansen aan het bedrijfslev...,"['adyen', 'google', 'x', 'kunstmatige intellig...",beleidsnota
1,Nota Defensie Industrie Strategie,2018,yes,[],Nederland wil zelf aan militaire kennisontwikk...,"['ai', 'artificiële intelligentie', 'drones', ...",beleidsnota
2,Nationale Strategie Digitaal Erfgoed 2021-2024,2021,yes,['google'],Zo wordt in de vernieuwde strategie nu ook de ...,"['google', 'ai', 'algoritmes', 'artificiële in...",beleidsnota
3,Beslisnota's bij de Kamerbrief over toekomstig...,2021,yes,['x'],25 januari\n37 9-2-2023 Nota-StasGB-Memo box 3...,"['x', 'ai']",beleidsnota
4,Beslisnota bij Kamerbrief over aanpak belastin...,2021,yes,['x'],Dit betreft de nota’s in de onderstaande tabel...,"['x', 'ai']",beleidsnota


In [25]:
tweede_kamer_data.head()

,title,year,ai_related,company_hits,body,matched_keywords_all,type
0,Investeren in Perspectief (Beleidsnota 2018),2018,yes,"['adyen', 'google', 'x']",Ook biedt de agenda kansen aan het bedrijfslev...,"['adyen', 'google', 'x', 'kunstmatige intellig...",beleidsnota
1,Nota Defensie Industrie Strategie,2018,yes,[],Nederland wil zelf aan militaire kennisontwikk...,"['ai', 'artificiële intelligentie', 'drones', ...",beleidsnota
2,Nationale Strategie Digitaal Erfgoed 2021-2024,2021,yes,['google'],Zo wordt in de vernieuwde strategie nu ook de ...,"['google', 'ai', 'algoritmes', 'artificiële in...",beleidsnota
3,Beslisnota's bij de Kamerbrief over toekomstig...,2021,yes,['x'],25 januari\n37 9-2-2023 Nota-StasGB-Memo box 3...,"['x', 'ai']",beleidsnota
4,Beslisnota bij Kamerbrief over aanpak belastin...,2021,yes,['x'],Dit betreft de nota’s in de onderstaande tabel...,"['x', 'ai']",beleidsnota


In [26]:
tweede_kamer_data.to_csv("tweede_kamer_data.csv")

In [27]:
tweede_kamer_data['matched_keywords_all'] = tweede_kamer_data['matched_keywords_all'].apply(
        lambda x: ast.literal_eval(x) if isinstance(x, str) else x
    )

In [77]:
# # make sure matched_keywords_all is list
# for idx, row in tweede_kamer_data.iterrows():
#     if isinstance(row['matched_keywords_all'], str):
#         try:
#             tweede_kamer_data.at[idx, 'matched_keywords_all'] = ast.literal_eval(row['matched_keywords_all'])
#         except (ValueError, SyntaxError):
#             tweede_kamer_data.at[idx, 'matched_keywords_all'] = []
# tweede_kamer_data['matched_keywords_all'].iloc[1]

In [28]:


def inspect_ai_related(df, num_samples=30, random_state=42):
    """
    Displays random samples with highlighted *matched* keywords.
    Uses df['matched_keywords_all'] per row instead of a global keyword list.
    """

    # --- sanity checks ---
    if 'ai_related' not in df.columns:
        raise KeyError("DataFrame must contain an 'ai_related' column.")
    if not {'title', 'body'}.issubset(df.columns):
        missing = {'title', 'body'} - set(df.columns)
        raise KeyError(f"Missing required column(s): {missing}")
    if 'matched_keywords_all' not in df.columns:
        raise KeyError("DataFrame must contain 'matched_keywords_all'.")

    # --- sample ---
    n = min(num_samples, len(df))
    samples = df.sample(n=n, random_state=random_state)

    # --- internal highlight function ---
    def highlight_keywords(text, keywords):
        if pd.isna(text):
            return ""
        s = str(text)

        if not isinstance(keywords, (set, list)):
            raise ValueError("Keywords must be a set or list")
        kws = {str(w).strip().lower() for w in keywords if str(w).strip()}
        if not kws:
            return s

        # sort by length to avoid partial overshadowing
        ordered = sorted(kws, key=len, reverse=True)

        # custom boundary: match even inside hyphenated words
        def make_pattern(word):
            return rf'(?<![A-Za-z0-9]){re.escape(word)}(?![A-Za-z0-9])'

        combined = "|".join(make_pattern(w) for w in ordered)
        regex = re.compile(combined, flags=re.IGNORECASE)

        def repl(m):
            kw = m.group(0)
            return f'<span style="font-size:1.5em; font-weight:bold; color:green;">{escape(kw)}</span>'

        return regex.sub(repl, s)

    # --- display samples ---
    for idx, row in samples.iterrows():
        ai_val = row['ai_related']
        title = row['title']
        body  = row['body']

        # --- use the actually matched keywords in this row ---
        matched_keywords = row.get('matched_keywords_all', [])
        if isinstance(matched_keywords, (list, set, tuple)):
            row_terms = {str(x).lower() for x in matched_keywords}
        else:
            row_terms = {str(matched_keywords).lower()} if matched_keywords else set()

        # --- highlight ---
        highlighted_title = highlight_keywords(title, row_terms)
        highlighted_body  = highlight_keywords(body, row_terms)

        # --- header ---
        header_html = (
            f'<div style="margin:0.5em 0;">'
            f'<strong>Index:</strong> {idx} &nbsp; | &nbsp; '
            f'<strong>ai_related:</strong> {ai_val}'
        )
        if matched_keywords:
            header_html += f'<br><strong>matched keywords:</strong> {matched_keywords}'
        header_html += '</div>'

        # --- display ---
        display(HTML(header_html))
        display(HTML(f'<h3 style="margin:0.2em 0;">{highlighted_title}</h3>'))
        display(HTML(f'<div style="line-height:1.5;">{highlighted_body}</div>'))
        display(HTML('<hr>'))

    print(f"Displayed {n} random articles (with row-specific matched keywords highlighted).")
    return samples

In [29]:
# rename relevant text to body
tweede_kamer_data = tweede_kamer_data.rename(columns={"relevant_text": "body"})
tweede_kamer_data = tweede_kamer_data.rename(columns={"matched_keywords": "matched_keywords_all"})

In [30]:
# Suppose df is your DataFrame
inspect_ai_related(tweede_kamer_data, num_samples=25, random_state=22)

Displayed 25 random articles (with row-specific matched keywords highlighted).


,title,year,ai_related,company_hits,body,matched_keywords_all,type
486,NaN,2017,yes,"['twitter', 'x']",Ik geef de heer Verhoeven namens D66 het woord...,"[twitter, x, algoritmen, algoritmes, artificië...",plenair verslag
488,NaN,2017,yes,['twitter'],De afgelopen jaren is onze samenleving al ingr...,"[twitter, kunstmatige intelligentie]",plenair verslag
372,Geannoteerde besluitenlijst ministerraad 1 maa...,2024,yes,[],d. Conclusies van de coördinatiecommissie d.d....,"[ai, kunstmatige intelligentie]",vergaderstuk
668,NaN,2022,yes,"['uber', 'x']",In de energiebelasting bijvoorbeeld zijn we al...,"[uber, x, ai, algoritmes]",plenair verslag
148,Beslisnota's bij Kamerbrieven over voornemen t...,2024,yes,['x'],Voorstellen:\nWoordvoeringslijn\n« Het budget ...,"[x, ai, kunstmatige intelligentie]",beleidsnota
456,NaN,2016,yes,['x'],"10, waarin wordt gevraagd om de termijn van ni...","[x, algoritmes]",plenair verslag
42,Beslisnota Verwerking advies Raad van State ov...,2023,yes,[],Het is volgens de\npolitie wel belangrijk om o...,"[ai, algoritmes, artificiële intelligentie]",beleidsnota
242,Beslisnota bij Kamerbrief met reactie op verzo...,2025,yes,[],Nota actief openbaar\nJa\nOnze referentie\nAan...,[ai],beleidsnota
5,Beslisnota Verlenging opdracht SON voor overbr...,2022,yes,['microsoft'],"De type werkzaamheden zijn gelijk, anders dan ...","[microsoft, azure]",beleidsnota
89,Beslisnota bij antwoorden Kamervragen over inf...,2023,yes,[],Raadsaanbeveling inzake toetreding tot het doo...,"[ai, artificial intelligence]",beleidsnota
